# R11 · Multi-core query timing (complete queries, every dataset)

How each step of a complete query scales with the number of physical cores T, on every dataset: the whole-matrix
logarithm (one dense eigendecomposition, T BLAS threads), the query's block logarithms (independent per niche and block,
on a pool of T worker processes), the whole-matrix and Spindle-naive scans (rows split over the pool) and Spindle's
search, both on one core (as R4) and split by niche over the pool (method-v3 `FlatSpindle.probe_niche` / `scan_niche`,
same output). Every parallel step is asserted equal, bit for bit, to its single-thread counterpart. At T = 1 the block
logs are also timed by the pool's per-block route on one core (the baseline of their scaling). Definitions:
`experiments/common/parallel_timing.py`. The paper's single-thread benchmark stays R4; T = 1 here repeats its protocol.

* **Prerequisites:** `R0_index_cache`, `R4_timing` (for the T = 1 check).
* **Mode:** environment variable `SPINDLE_PARALLEL` = `pilot` (brain only, seed 0, 20 queries, T = 1, 8, 64; always
  rewritten) or `full` (every dataset, seeds 0–4, first 200 held-out queries, T = 1, 2, 4, 8, 16, 32, 64).
* **Writes:** `results/parallel/`. Per dataset: `per_dataset/<key>/timing.csv` (one row per T, seed, query: ms of every
  step). Pooled: `timing.csv`, `seeds.csv` (median per dataset, T, seed) and `summary.csv` (mean and s.d. over seeds,
  end-to-end totals, scaling over T = 1, end-to-end speed-ups of Spindle). Existing per-dataset files are skipped.
* **Run:** `sbatch --export=ALL,SPINDLE_PARALLEL=full experiments/common/parallel_nb.sbatch experiments/compute/R11_parallel.ipynb`
  (65 physical cores of ROSALIND-CPU1: 64 workers + the main process, one hardware thread per core; workers busy-wait,
  see parallel_timing.py; the node is shared with other users' jobs). About 1–1.5 h; peak memory about 10 GB (brain).


In [1]:
import gc
import os
import subprocess
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets.yaml").exists())
sys.path.insert(0, str(ROOT / "experiments" / "common"))
import pandas as pd
from threadpoolctl import threadpool_info

import data
import parallel_timing as pt
import results_io as rio

TOPIC = "parallel"
MODE = os.environ.get("SPINDLE_PARALLEL", "pilot")
if MODE == "pilot":
    KEYS, SEEDS, N_QUERIES, THREADS, NAME, FORCE = ["brain_cancer"], [0], 20, [1, 8, 64], "pilot_timing.csv", True
else:
    KEYS, SEEDS, N_QUERIES, THREADS, NAME, FORCE = data.indexed_keys(), data.SEEDS, 200, [1, 2, 4, 8, 16, 32, 64], "timing.csv", False
print(data.stamp(), "| mode", MODE, "| job", os.environ.get("SLURM_JOB_ID"), "| cpus", len(os.sched_getaffinity(0)))
print([(i["internal_api"], i["version"], i["threading_layer"], i["num_threads"]) for i in threadpool_info()])
print(subprocess.run(["bash", "-c", "cat /proc/loadavg; squeue -h -w $(hostname -s) -o '%i %u %C %j' | head -40"],
                     capture_output=True, text=True).stdout)

{'tag': 'method-v2', 'describe': 'method-v3-dirty', 'commit': 'b35f7d4'} | mode full | job None | cpus 16
[('openblas', '0.3.34', 'pthreads', 16), ('openblas', '0.3.31.dev', 'pthreads', 16)]
1.60 0.79 0.42 5/1817 3033500



## Time every dataset (skipped where its file exists)

In [2]:
for key in KEYS:
    f = rio.per_dataset(TOPIC, key) / NAME
    if f.exists() and not FORCE:
        print(f"{key}: exists, skipped")
        continue
    df = pt.dataset_parallel(key, THREADS, N_QUERIES, SEEDS, log=lambda m: print(m, flush=True))
    df.to_csv(f, index=False)
    rio.register(TOPIC, f, f"{key}: one row per (threads, seed, query), ms of every step on T physical cores ({MODE})",
                 len(df))
    del df
    gc.collect()

skin_melanoma: exists, skipped
kidney_nondiseased: exists, skipped
breast_cancer: exists, skipped
lung_cancer: exists, skipped
pancreatic_cancer: exists, skipped
lymph_node: exists, skipped
lymph_node_5k: exists, skipped
brain_cancer: exists, skipped


## Pooled tables

In [3]:
tm = pd.concat([pd.read_csv(rio.topic(TOPIC) / "per_dataset" / k / NAME) for k in KEYS], ignore_index=True)
seeds, summ = pt.summary(tm)
pre = "pilot_" if MODE == "pilot" else ""
rio.write_csv(tm, TOPIC, f"{pre}timing.csv", "one row per (dataset, threads, seed, query): ms of every step on T cores")
rio.write_csv(seeds, TOPIC, f"{pre}seeds.csv", "median ms of every step and end-to-end total per (dataset, threads, seed)")
rio.write_csv(summ, TOPIC, f"{pre}summary.csv", "mean and s.d. over seeds of the per-seed medians, end-to-end totals, "
              "scaling over T = 1, end-to-end speed-ups of Spindle")

wrote results/parallel/timing.csv: 46935 rows
wrote results/parallel/seeds.csv: 280 rows
wrote results/parallel/summary.csv: 56 rows


PosixPath('/home/NAShome/ghoss18/spindle_dev/results/parallel/summary.csv')

## Summary and the T = 1 check against R4

In [4]:
pd.set_option("display.width", 250)
cols = ["key", "threads", "e2e_whole_ms", "e2e_naive_ms", "e2e_spindle_ms", "speedup_e2e_spindle_vs_whole",
        "speedup_e2e_spindle_vs_naive", "qlog_whole_ms", "qlog_block_ms", "spindle_par_ms"]
display(summ[cols].round(2))
r4 = pd.read_csv(ROOT / "results" / "timing" / "whole_tile.csv")
r4 = r4[r4.key.isin(KEYS) & r4.seed.isin(SEEDS) & (r4["query"] < N_QUERIES)]
steps = [c for c in pt.STEPS if c in r4.columns]
r4m = r4.groupby(["key", "seed"])[steps].median().groupby("key").mean()
one = seeds[seeds.threads == 1].groupby("key")[steps].mean()
display((one / r4m).round(3).rename(columns=lambda c: c + " (R11 T=1 / R4)"))

,key,threads,e2e_whole_ms,e2e_naive_ms,e2e_spindle_ms,speedup_e2e_spindle_vs_whole,speedup_e2e_spindle_vs_naive,qlog_whole_ms,qlog_block_ms,spindle_par_ms
0,brain_cancer,1,456.50,41.29,28.63,15.95,1.44,14.25,25.61,3.04
1,brain_cancer,2,237.29,17.31,10.90,21.77,1.59,12.86,8.91,2.00
2,brain_cancer,4,125.40,9.70,6.20,20.23,1.57,11.81,4.71,1.50
3,brain_cancer,8,93.45,7.00,3.78,24.76,1.86,10.90,2.56,1.19
4,brain_cancer,16,72.51,5.15,2.54,28.52,2.02,11.21,1.50,1.04
5,brain_cancer,32,44.61,3.38,2.06,21.70,1.65,12.07,0.99,1.06
6,brain_cancer,64,156.25,2.50,2.03,76.86,1.23,135.47,0.78,1.24
7,breast_cancer,1,29.35,5.36,4.05,7.25,1.32,5.36,3.65,0.41
8,breast_cancer,2,17.54,2.67,1.98,8.85,1.34,5.22,1.68,0.30
9,breast_cancer,4,13.82,1.59,1.13,12.27,1.41,5.84,0.88,0.24


,qlog_whole_ms (R11 T=1 / R4),qlog_block_ms (R11 T=1 / R4),whole_ms (R11 T=1 / R4),naive_ms (R11 T=1 / R4),spindle_ms (R11 T=1 / R4)
key,,,,,
brain_cancer,0.972,1.005,1.002,1.038,1.046
breast_cancer,0.962,0.964,1.006,1.005,0.988
kidney_nondiseased,0.972,0.968,1.006,0.995,0.906
lung_cancer,0.957,0.966,1.007,0.994,0.992
lymph_node,0.961,0.976,0.992,1.015,1.014
lymph_node_5k,0.961,0.989,1.003,1.052,1.001
pancreatic_cancer,0.968,0.976,1.002,1.027,1.029
skin_melanoma,0.975,0.983,1.001,0.984,0.988


In [5]:
rio.write_manifest(TOPIC, "R11_parallel.ipynb", inputs=["results/_cache (R0_index_cache)", "results/timing (R4_timing)"],
                   notes=f"every dataset on T physical cores of ROSALIND-CPU1 (one hardware thread per core; node shared "
                         f"with other users' jobs); mode {MODE}. The brain-only runs of 4 Nov are in _brain_only_2026-11-04/.")

wrote results/parallel/manifest.json: 11 files; peak memory 0.15 GB


{'topic': 'parallel',
 'notebook': 'R11_parallel.ipynb',
 'date': '2026-11-04 23:59',
 'tag': 'method-v2',
 'describe': 'method-v3-dirty',
 'commit': 'b35f7d4',
 'inputs': ['results/_cache (R0_index_cache)', 'results/timing (R4_timing)'],
 'peak_rss_gb': 0.15,
 'notes': "every dataset on T physical cores of ROSALIND-CPU1 (one hardware thread per core; node shared with other users' jobs); mode full. The brain-only runs of 4 Nov are in _brain_only_2026-11-04/.",
 'files': {'per_dataset/brain_cancer/timing.csv': {'description': 'brain_cancer: one row per (threads, seed, query), ms of every step on T physical cores (full)',
   'rows': 7000,
   'bytes': 1552376},
  'per_dataset/breast_cancer/timing.csv': {'description': 'breast_cancer: one row per (threads, seed, query), ms of every step on T physical cores (full)',
   'rows': 7000,
   'bytes': 1561107},
  'per_dataset/kidney_nondiseased/timing.csv': {'description': 'kidney_nondiseased: one row per (threads, seed, query), ms of every step o